# Downloading and Loading Training Data from Zenodo for TalentCLEF (Task-B)

This notebook provides a step-by-step guide on downloading and loading training data for the TalentCLEF shared-task B, hosted on [Zenodo](https://doi.org/10.5281/zenodo.14002665).

TalentCLEF is an initiative to advance Natural Language Processing (NLP) in Human Capital Management (HCM). It aims to create a public benchmark for model evaluation and promote collaboration to develop fair, multilingual, and flexible systems that improve Human Resources (HR) practices acro
ss different industries.

This shared-task's inaugural edition is part of the [Conference and Labs of the Evaluation Forum (CLEF)](https://clef2025.clef-initiative.eu/index.php?page=Pages/labs.html), scheduled to be held in Madrid in 2025. If you are interested in registering, you can find registration form [here](https://clef2025-labs-registration.dei.unipd.it/).

<img src="https://github.com/TalentCLEF/talentclef/blob/main/logo_talentclef.png?raw=true" alt="TalentCLEF logo" width="200"/>
<img src="https://talentclef.github.io/talentclef/docs/talentclef-2025/workshop/logo_clef_madrid.png" alt="TalentCLEF logo" width="150"/>

In this notebook you will learn how to download and load the training data of the task.

## Import Required Libraries

In [3]:
import os
import sys
import pandas as pd
import json

## Download files (Task B)

<!--First, let's download the Task A and Task B zip files directly from Zenodo.-->
First, let's download the Task B zip files directly from Zenodo.

In [7]:
# Download

!curl --output TaskB.zip https://zenodo.org/records/14693201/files/TaskB.zip

# If curl command does not work, please uncomment (#) the following wget command and run the cell again
# !wget https://zenodo.org/records/14693201/files/TaskB.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4093k  100 4093k    0     0  1439k      0  0:00:02  0:00:02 --:--:-- 1439k
Archive:  TaskB.zip
   creating: taskB/test
   creating: taskB/training
  inflating: taskB/training/job2skill.tsv  
  inflating: taskB/training/jobid2terms.json  
  inflating: taskB/training/skillid2terms.json  
   creating: taskB/validation


In [ ]:
# Unzip
!unzip TaskB.zip -d taskB

## Preparing data (Task B)

### Training data dictionary

The training data for Task B is shared in 3 files:
- `job2skill.tsv`: This file has been curated to include the most representative skills for each job title in <a href="https://esco.ec.europa.eu/en">ESCO</a>. This file contains three columns:
  - `job_id`: ESCO identifier for the job position.
  - `skill_id`: ESCO identifier for the skill.
  - `rel_type`: Indicator specifying whether the skill_id is essential or optional for a specific job_id. It can have the value “essential” or “optional.”

- `jobid2terms.json`: This JSON file contains job_id identifiers used in the training set for Task B as keys, and a list of valid lexical variants for each identifier as values.

- `skillid2terms.json`: This JSON file contains skill_id identifiers as keys, and a list of valid lexical variants for each identifier as values.



#### Setting up data path

In [9]:
taskB_path="taskB"

In [10]:
taskB_training_path = os.path.join(taskB_path, "training")

#### Reading Job2Skill data file

In [13]:
# Read job2skill file
job2skill = pd.read_csv(os.path.join(taskB_training_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])

print ("No. of job_id and skill_id triple: {}".format(job2skill.shape[0]))
job2skill.head()

No. of job_id and skill_id triple: 114699


,job_id,skill_id,rel_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential


In [14]:
job2skill['skill_id'][:10]

0    http://data.europa.eu/esco/skill/93a68dcb-3dc6...
1    http://data.europa.eu/esco/skill/05bc7677-5a64...
2    http://data.europa.eu/esco/skill/860be36a-d19b...
3    http://data.europa.eu/esco/skill/fed5b267-73fa...
4    http://data.europa.eu/esco/skill/f64fe2c2-d090...
5    http://data.europa.eu/esco/skill/271a36a0-bc7a...
6    http://data.europa.eu/esco/skill/591dd514-735b...
7    http://data.europa.eu/esco/skill/47ed1d37-971b...
8    http://data.europa.eu/esco/skill/892f8e2f-189a...
9    http://data.europa.eu/esco/skill/a2dfc063-7b24...
Name: skill_id, dtype: object

#### Reading Job2Terms file and Skilled2Terms

#### Reading Job2Terms file and Skilled2Terms

In [ ]:
# Read json files
jobid2terms_path = os.path.join(taskB_training_path, "jobid2terms.json")
print (jobid2terms_path)

In [ ]:
with open(os.path.join(taskB_training_path, "jobid2terms.json"), 'r') as file:
    jobid2terms = json.load(file)

In [ ]:
len(jobid2terms)

In [ ]:
with open(os.path.join(taskB_training_path, "skillid2terms.json"), 'r') as file:
    skillid2terms = json.load(file)
len(skillid2terms)

#### Expanding Job2Skill with Terms Mapping

In [ ]:
job2skill["job_terms"] = job2skill["job_id"].map(jobid2terms)
job2skill["skill_terms"] = job2skill["skill_id"].map(skillid2terms)

In [ ]:
job2skill.describe()

**Let's see the skills for a specific job_id.**



In [ ]:
job_id_to_search = job2skill.job_id.to_list()[100]
job_data = job2skill[job2skill.job_id == job_id_to_search]

**The job titles of the selected job_id are:**

In [ ]:
job_data.job_terms.iloc[0]

**For that job_id we fin 9 essential skills and 1 optional skill:**

In [ ]:
job_data.rel_type.value_counts()

The list of skills related to the job_id are:

In [ ]:
job_data.skill_terms.to_list()

In [ ]:
job2skill.job_id.to_list()[100]